In [29]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Importuri**
- `torch` — biblioteca principală (tensori + diferențiere automată).
- `torch.nn as nn` — stratul de "cărămizi" pentru rețele neuronale (`Linear`, `Embedding`, `LayerNorm`, `Dropout`...).
- `torch.nn.functional as F` — funcții fără parametri învățabili (`softmax`, `cross_entropy`).

**Hiperparametri** (setări pe care *le alegi tu*, nu sunt învățate de model):
| Nume | Ce înseamnă |
|---|---|
| `batch_size = 64` | Câte secvențe independente procesăm simultan la un pas de antrenare. Mai mare = mai stabil, dar consumă mai multă memorie. |
| `block_size = 256` | Lungimea maximă a contextului: modelul se uită la cel mult 256 de caractere anterioare ca să prezică următorul. |
| `max_iters = 5000` | Câți pași de antrenare facem. |
| `eval_interval = 500` | La fiecare câți pași afișăm loss-ul. |
| `learning_rate = 3e-4` | Cât de mari sunt "pașii" de actualizare a greutăților. Prea mare = instabil, prea mic = lent. |
| `device` | `'cuda'` dacă ai GPU, altfel `'cpu'`. |
| `eval_iters = 200` | Câte batch-uri folosim ca să *estimăm* loss-ul (mediem ca să reducem zgomotul). |
| `n_embd = 384` | Dimensiunea vectorului care reprezintă fiecare caracter (câte numere descriu un token). |
| `n_head = 6` | Numărul de "capete" de atenție în paralel. |
| `n_layer = 6` | Câte blocuri Transformer punem unul după altul. |
| `dropout = 0.2` | Probabilitatea de a "stinge" aleator neuroni în antrenare, ca să evităm overfitting-ul. |

`torch.manual_seed(47)` fixează aleatoriul ca rezultatele să fie reproductibile.

In [30]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hiperparametri
batch_size = 64
block_size = 256
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2

torch.manual_seed(47)
print("Rulăm pe:", device)

Rulăm pe: cuda



Un model lucrează cu **numere**, nu cu litere. Facem un tokenizator foarte simplu, la nivel de caracter:

- `chars` = lista sortată a tuturor caracterelor *unice* din text (litere, spațiu, punctuație, `\n`...). Sunt ~65.
- `vocab_size` = câte caractere unice avem (dimensiunea "vocabularului").
- `stoi` ("string to int") = dicționar caracter → număr.
- `itos` ("int to string") = dicționar invers, număr → caracter.
- `encode("..")` → listă de numere, `decode(...)` →

Modelele reale (GPT-4 etc.) folosesc tokenizatori pe *subcuvinte* (BPE), cu zeci de mii de tokeni; aici vocabularul mic păstrează totul simplu.

In [31]:
base_path='/content/drive/MyDrive/03_Proiecte/LLM_Scratch'
file=base_path + '/input.txt'
with open(file, 'r', encoding = 'utf-8') as f:
  text=f.read()

# unique characters from the text
chars=list(set(text))
print(chars)
vocab_size=len(chars)

#tokenizer
stoi={val:index  for index,val in enumerate(chars)}
itos = {index:val for index,val in enumerate(chars)}

print(stoi)
print(itos)

#endcoder (String to Integer)
def encoder(text):
  encoder=[]
  for i in text:
    encoder.append(stoi[i])
  return encoder

#decoder (Integer to String)
def decoder(text):
  decoder=[]
  for i in text:
    decoder.append(itos[i])

  return decoder

codat=encoder('Ana are mere !$&')
decoder(codat)


['O', 'S', 'F', 'Y', 'f', 'R', ' ', 'w', 'U', 'h', 'C', 'W', "'", 'x', 'g', '3', 'D', 'e', 'r', 'z', ';', 'E', 'H', 'T', 'G', 's', 'y', 'M', 'm', 'i', 'B', 'k', 'L', 'I', 'q', 'J', 'n', 'l', 'V', 'Q', '&', ':', 'A', '$', 'j', 'X', '\n', 'P', 'p', 'd', 'u', 'Z', 'v', 't', ',', 'N', '!', 'b', '?', 'K', '-', 'o', 'c', '.', 'a']
{'O': 0, 'S': 1, 'F': 2, 'Y': 3, 'f': 4, 'R': 5, ' ': 6, 'w': 7, 'U': 8, 'h': 9, 'C': 10, 'W': 11, "'": 12, 'x': 13, 'g': 14, '3': 15, 'D': 16, 'e': 17, 'r': 18, 'z': 19, ';': 20, 'E': 21, 'H': 22, 'T': 23, 'G': 24, 's': 25, 'y': 26, 'M': 27, 'm': 28, 'i': 29, 'B': 30, 'k': 31, 'L': 32, 'I': 33, 'q': 34, 'J': 35, 'n': 36, 'l': 37, 'V': 38, 'Q': 39, '&': 40, ':': 41, 'A': 42, '$': 43, 'j': 44, 'X': 45, '\n': 46, 'P': 47, 'p': 48, 'd': 49, 'u': 50, 'Z': 51, 'v': 52, 't': 53, ',': 54, 'N': 55, '!': 56, 'b': 57, '?': 58, 'K': 59, '-': 60, 'o': 61, 'c': 62, '.': 63, 'a': 64}
{0: 'O', 1: 'S', 2: 'F', 3: 'Y', 4: 'f', 5: 'R', 6: ' ', 7: 'w', 8: 'U', 9: 'h', 10: 'C', 11: 'W

['A',
 'n',
 'a',
 ' ',
 'a',
 'r',
 'e',
 ' ',
 'm',
 'e',
 'r',
 'e',
 ' ',
 '!',
 '$',
 '&']


- `data` = tot textul transformat într-un tensor de numere întregi (`dtype=torch.long`).
- Primele 90% (`train_data`) servesc la **antrenare**.
- Ultimele 10% (`val_data`) **nu sunt văzute la antrenare**; le folosim doar ca să verificăm dacă modelul chiar generalizează sau doar memorează.

Dacă loss-ul pe train scade, dar cel pe val crește, ai **overfitting**.

In [32]:
data=torch.tensor(encoder(text), dtype=torch.long)
print(data.shape, data.dtype)

n=int(0.9*len(text))
train_data=data[:n]
test_data=data[n:]

print(len(train_data))
print(len(test_data))

torch.Size([1115394]) torch.int64
1003854
111540


##Încărcarea datelor (`get_batch`)

Ce face, pas cu pas:
1. Alege `train_data` sau `val_data` după `split`.
2. `ix = torch.randint(len(data) - block_size, (batch_size,))` → alege **64 de poziții aleatoare** de start în text.
3. `x` = pentru fiecare poziție `i`, bucata `data[i : i+block_size]` (256 caractere). `torch.stack` le pune într-un tensor `(64, 256)`.
4. `y` = aceeași bucată, dar **deplasată cu 1**: `data[i+1 : i+block_size+1]`. Deci `y[t]` este caracterul care urmează după `x[t]`.
5. Mutăm tensorii pe GPU (`.to(device)`).

**Exemplu:** dacă `x = "Hell"`, atunci `y = "ello"`. Din *o singură* bucată de 256 de caractere obținem 256 de exemple de antrenare: din "H" prezice "e", din "He" prezice "l", din "Hel" prezice "l"...[link text](https://)

In [33]:
def get_batch(split):

  if split == 'train':
    data=train_data
  else:
    data = test_data

  #we take random pozitions so the model doesn t learn from patterns
  pozitii=torch.randint(
      0,len(data) - block_size, (batch_size,)
  )

  x= []
  y = []

  for pozitie in pozitii:
    x.append(data[pozitie : pozitie+block_size])
    y.append ( data[pozitie+1 : pozitie+block_size+1])

  x = torch.stack(x)
  y = torch.stack(y)

  return x, y


xb, yb = get_batch('train')

print("INPUT")
print(xb)

print("TARGET")
print(yb)

print("SHAPE")
print(xb.shape, yb.shape)

INPUT
tensor([[49,  6, 28,  ..., 29, 36,  6],
        [ 9, 17, 18,  ..., 50, 18, 49],
        [ 6, 29, 53,  ..., 61,  4,  6],
        ...,
        [28, 48, 64,  ..., 46, 42,  6],
        [18,  6, 62,  ...,  9, 64, 53],
        [33,  6, 17,  ..., 18, 29, 36]])
TARGET
tensor([[ 6, 28, 64,  ..., 36,  6, 48],
        [17, 18,  6,  ..., 18, 49, 17],
        [29, 53,  6,  ...,  4,  6, 36],
        ...,
        [48, 64, 25,  ..., 42,  6, 57],
        [ 6, 62, 18,  ..., 64, 53,  6],
        [ 6, 17, 13,  ..., 29, 36, 62]])
SHAPE
torch.Size([64, 256]) torch.Size([64, 256])


##Estimarea loss-ului (`estimate_loss`)

- `@torch.no_grad()` — nu calculăm gradienți aici (economisim memorie și timp, pentru că doar evaluăm).
- `model.eval()` — trece modelul în mod evaluare (dezactivează dropout-ul). La final, `model.train()` îl readuce în modul antrenare.
- Pentru `train` și `val`, rulăm 200 de batch-uri și facem **media** loss-ului. Un singur batch ar fi prea zgomotos.

Funcția folosește variabila `model`, pe care o definim mai jos, dar nu e o problemă: Python o caută abia când funcția e *apelată*.

In [34]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

##Un cap de self-attention (`Head`) — inima Transformer-ului

**Intuiția:** fiecare caracter din secvență "se uită" la caracterele anterioare și decide pe care să le asculte mai mult. Fiecare token produce 3 vectori:
- **query (q)** — "ce caut eu?"
- **key (k)** — "ce conțin eu?"
- **value (v)** — "dacă te interesez, asta îți dau"

**Pașii din `forward`:**
1. `B, T, C = x.shape` → B = batch (64), T = timp/poziții (≤256), C = canale/embedding (384).
2. `k`, `q`, `v` se obțin cu straturi `Linear` (fără bias), fiecare reduce de la `C` la `head_size`.
3. `wei = q @ k.transpose(-2,-1)` → matrice `(T, T)` de **scoruri de afinitate**: cât de mult se potrivește query-ul fiecărui token cu key-ul fiecărui alt token.
4. `* k.shape[-1]**-0.5` → împărțim la √head_size. Fără asta, valorile mari fac softmax-ul prea "ascuțit" și antrenarea suferă.
5. **Masca cauzală:** `tril` e o matrice triunghiulară inferioară (1 sub diagonală, 0 deasupra). `masked_fill(... == 0, -inf)` pune `-inf` în viitor, așa că un token **nu poate vedea tokenii de după el** (altfel ar "copia" răspunsul).
6. `softmax` transformă scorurile în probabilități (fiecare rând însumează 1; `-inf` devine 0).
7. `dropout` pe ponderile de atenție (regularizare).
8. `out = wei @ v` → media ponderată a value-urilor. Asta e "informația adunată" de la tokenii anteriori.

`register_buffer('tril', ...)` — salvează matricea ca parte din model (se mută pe GPU odată cu el), dar **nu e parametru antrenabil**.

In [35]:
class Head(nn.Module):
    """ un cap de self-attention """
    def __init__(self, head_size):
        super().__init__()
        #Fiecare linie creează doar o matrice de greutăți (un strat liniar) de formă 384 → 64, inițializată cu numere aleatoare mici:
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape  # x ul are (64,256,384)
        k = self.key(x)    # (B,T,hs)
        q = self.query(x)  # (B,T,hs)
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5      # (B,T,T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)                           # (B,T,T)
        wei = self.dropout(wei)
        v = self.value(x)                                      # (B,T,hs)
        out = wei @ v                                          # (B,T,hs)
        return out

In [36]:
T = 4
tril = torch.tril(torch.ones(T, T))
scores = torch.randn(T, T)
masked = scores.masked_fill(tril == 0, float('-inf'))
print("scoruri cu mască:\n", masked)
print("după softmax:\n", F.softmax(masked, dim=-1))
# rândul 0 se uită doar la el; rândul 3 se uită la toți 4. Nimeni nu vede viitorul.

scoruri cu mască:
 tensor([[-0.4320,    -inf,    -inf,    -inf],
        [-0.1891, -0.3025,    -inf,    -inf],
        [-1.5224, -1.5398,  0.4258,    -inf],
        [-0.2197,  1.1220, -0.6992,  1.7546]])
după softmax:
 tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5283, 0.4717, 0.0000, 0.0000],
        [0.1111, 0.1092, 0.7797, 0.0000],
        [0.0791, 0.3025, 0.0490, 0.5695]])


## Celula 8 — Multi-head attention

Un singur cap învață un singur tip de relații. Rulăm **mai multe capete în paralel** (`n_head = 6`), fiecare cu `head_size = n_embd // n_head = 64`, ca să prindă relații diferite (ex: un cap urmărește rima, altul punctuația...).

- `nn.ModuleList([...])` — listă de module pe care PyTorch le "vede" (le înregistrează parametrii).
- `torch.cat([...], dim=-1)` — lipim ieșirile celor 6 capete: 6 × 64 = 384 de canale.
- `self.proj` — strat liniar care amestecă informația din capete și o aduce înapoi la `n_embd`.
- `dropout` la final.

In [37]:
class MultiHeadAttention(nn.Module):
    """ mai multe capete de self-attention în paralel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

## Celula 9 — Feed-forward (MLP)

După ce tokenii au "comunicat" prin atenție, fiecare token trece **individual** printr-un mic MLP ("gândește" la ce a aflat):
- `Linear(n_embd → 4*n_embd)` — extinde dimensiunea de 4 ori (convenția din articolul "Attention Is All You Need").
- `ReLU` — non-linearitate (fără ea, mai multe straturi liniare ar fi echivalente cu unul singur).
- `Linear(4*n_embd → n_embd)` — revine la dimensiunea inițială.
- `Dropout` — regularizare.

(Numele clasei `FeedFoward` are o greșeală de scriere în codul original; e inofensivă, dar o poți scrie `FeedForward` dacă o schimbi peste tot.)

In [38]:
class FeedFoward(nn.Module):
    """ un strat liniar urmat de o non-liniaritate """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

## Celula 10 — Blocul Transformer

Un bloc = **comunicare** (atenție) + **calcul** (feed-forward). Două trucuri esențiale pentru rețele adânci:

- **Conexiuni reziduale:** `x = x + self.sa(...)`. Adunăm intrarea la ieșire, ca gradienții să curgă ușor înapoi prin multe straturi. Fără ele, antrenarea a 6 blocuri ar merge mult mai greu.
- **LayerNorm:** normalizează valorile fiecărui token (medie 0, varianță 1) înainte de atenție și de MLP. Stabilizează antrenarea. (Aici e varianta "pre-norm", folosită în GPT-2/3.)

`head_size = n_embd // n_head` = 384 // 6 = 64.

In [39]:
class Block(nn.Module):
    """ bloc Transformer: comunicare urmată de calcul """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

## Celula 11 — Modelul GPT complet

**În `__init__`:**
- `token_embedding_table = nn.Embedding(vocab_size, n_embd)` — tabel care transformă fiecare id de caracter într-un vector de 384 numere (învățat).
- `position_embedding_table = nn.Embedding(block_size, n_embd)` — la fel, dar pentru **poziție** (0..255). Atenția în sine nu știe ordinea tokenilor, deci îi spunem explicit unde se află fiecare.
- `blocks` — 6 blocuri Transformer puse în serie (`nn.Sequential`; `*[...]` despachetează lista).
- `ln_f` — LayerNorm final.
- `lm_head = nn.Linear(n_embd, vocab_size)` — transformă vectorul final în **un scor (logit) pentru fiecare caracter posibil** din vocabular.
- `self.apply(self._init_weights)` — inițializează greutățile cu o normală mică (std 0.02), cum face GPT-2, pentru o pornire stabilă.

**În `forward(idx, targets)`:**
1. `idx` are forma `(B, T)` — id-uri de caractere.
2. `tok_emb + pos_emb` — adunăm "ce caracter e" + "unde e" → `(B, T, C)`.
3. Trecem prin blocuri → `ln_f` → `lm_head` → `logits` de formă `(B, T, vocab_size)`.
4. Dacă avem `targets`, calculăm **cross-entropy**: măsoară cât de puțină probabilitate a dat modelul caracterului corect. Pentru asta aplatizăm `(B,T,C)` în `(B*T, C)`, pentru că `F.cross_entropy` așteaptă forma asta.

**`generate(idx, max_new_tokens)` — cum "scrie" modelul:**
1. Tăiem contextul la ultimele `block_size` caractere (`idx[:, -block_size:]`), pentru că pozițiile >255 nu există în tabel.
2. Rulăm modelul și păstrăm doar logits-urile **ultimei poziții** (`[:, -1, :]`) — predicția pentru următorul caracter.
3. `softmax` → probabilități.
4. `torch.multinomial` **trage la sorți** un caracter după acele probabilități (de aceea textul diferă la fiecare rulare).
5. Îl lipim la secvență (`torch.cat`) și repetăm.

In [40]:
class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)                                # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))  # (T,C)
        x = tok_emb + pos_emb                                                    # (B,T,C)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                                                 # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

## Celula 12 — Creăm modelul și optimizatorul

- `GPTLanguageModel()` creează modelul; `.to(device)` îl mută pe GPU.
- `p.numel()` = numărul de elemente al fiecărui parametru; le însumăm → ~10.8 milioane de parametri (un GPT-3 are 175 de miliarde).
- **AdamW** = optimizatorul care actualizează greutățile pe baza gradienților (Adam + weight decay corect implementat). `lr` = learning rate-ul definit sus.

In [41]:
model = GPTLanguageModel()
m = model.to(device)
print(sum(p.numel() for p in m.parameters()) / 1e6, 'M parameters')

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

10.788929 M parameters


## Celula 13 — Bucla de antrenare

La fiecare iterație:
1. **(din când în când)** evaluăm și afișăm loss-ul pe train și val.
2. `get_batch('train')` — luăm un batch nou.
3. `model(xb, yb)` — **forward pass**: predicții + loss.
4. `optimizer.zero_grad(set_to_none=True)` — ștergem gradienții de la pasul anterior (altfel s-ar aduna).
5. `loss.backward()` — **backpropagation**: calculează gradientul loss-ului față de fiecare parametru.
6. `optimizer.step()` — modifică ușor parametrii în direcția care scade loss-ul.

**La ce să te aștepți:** loss-ul pornește de la ~4.2 (≈ `ln(65)`, adică alegere complet aleatoare) și ajunge pe la ~1.5 pe val după 5000 de pași. Pe un T4 durează în jur de 15–30 minute.

In [42]:
# Redefining get_batch to ensure tensors are on the correct device
def get_batch(split):
    data = train_data if split == 'train' else test_data
    pozitii = torch.randint(0, len(data) - block_size, (batch_size,))
    x = torch.stack([data[pozitie : pozitie+block_size] for pozitie in pozitii])
    y = torch.stack([data[pozitie+1 : pozitie+block_size+1] for pozitie in pozitii])
    return x.to(device), y.to(device)

for iter in range(max_iters):

    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.1626, val loss 4.1595
step 500: train loss 1.7616, val loss 1.9170
step 1000: train loss 1.3982, val loss 1.6055
step 1500: train loss 1.2707, val loss 1.5317
step 2000: train loss 1.1904, val loss 1.4882
step 2500: train loss 1.1285, val loss 1.4797
step 3000: train loss 1.0756, val loss 1.4793
step 3500: train loss 1.0159, val loss 1.4918
step 4000: train loss 0.9675, val loss 1.5022
step 4500: train loss 0.9193, val loss 1.5371
step 4999: train loss 0.8624, val loss 1.5650


## Celula 14 — Generăm text

- `context = torch.zeros((1, 1), dtype=torch.long)` — pornim de la un singur token cu id 0 (în Shakespeare, id-ul 0 e caracterul newline `\n`). Batch de 1, lungime 1.
- `m.generate(context, max_new_tokens=500)` returnează `(1, 501)` id-uri; `[0]` ia primul (singurul) rând, `.tolist()` îl face listă, `decode` îl face text.

Linia comentată din codul original salvează 10.000 de caractere într-un fișier `more.txt`; o activezi mai jos dacă vrei.

In [44]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(''.join(decoder(m.generate(context, max_new_tokens=500)[0].tolist())))

OUS:
O heavy heavy blood hat, I know, our leleve.

KING RICHAR II:
You well?

TRANIO:
Bid me some senses of jumpet the queen your mind,
Your honour tell me what you are.

BALTHASARENCE:
Who knows not?

BISHERBARNARDINE:
To Great at unhappiness, unlessenced here;
But you do yourself hear with these things old
To humble warning put for mouring; though she dew
What think ushSe ours: but your treacher, abord for yourself
Maid; whilst I take him
He makes a walnant to your soul. You have so
That wrough
